# Human-reviewed follow-up of the Bio-Babel pancreas analysis

This notebook begins from the frozen outputs produced by the single-prompt
Claude Code analysis in `agent.ipynb`. It does **not** repeat manual
preprocessing, DDRTree reconstruction, BEAM testing or the initial tradeSeq
model fits. It performs the subsequent human review, program-level summaries,
candidate-screen interpretation and publication plotting.

The division of work is therefore explicit:

- `preprocess.ipynb`: manual quality control and annotation;
- `agent.ipynb`: agent-written and executed downstream analysis;
- `followup.ipynb`: human-reviewed follow-up using the agent outputs below.


## Frozen inputs

The trajectory checkpoint is intentionally not committed to Git. Place the
deposited file at `data/traj_ab.h5ad`, or set
`BIOBABEL_TRAJECTORY_INPUT=/path/to/traj_ab.h5ad`. Its SHA-256 is recorded in
`provenance.json`.

The repository tables used here are:

- `tables/beam.csv.gz`: agent-generated BEAM results;
- `tables/fig2g.tsv`: agent-fitted tradeSeq smoother values;
- `tables/screen_control.tsv.gz`: completed 1,116-gene condition screen;
- `tables/sfig6g.tsv` and `tables/sfig6g_tests.tsv`: embryo allocation values;
- `tables/sfig6h.tsv`: cross-lineage genotype-effect display table.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import os
import random
import subprocess
import sys

import anndata as ad
import ggplot2_py as gg
import grid_py
import igraph as ig
import monocle2py as m2
import numpy as np
import pandas as pd
import pheatmap
import scipy.sparse as sp
from grid_py.renderer import CairoRenderer
from IPython.display import Image, display
from scipy import stats


def find_repo_root(start):
    override = os.environ.get("BIOBABEL_PANCREAS_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "tables").is_dir() and (
            candidate / "agent.ipynb"
        ).is_file():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the bio-babel-pancreas repository, or set "
        "BIOBABEL_PANCREAS_ROOT."
    )


REPO_ROOT = find_repo_root(Path.cwd())
TABLES = REPO_ROOT / "tables"
sys.path.insert(0, str(REPO_ROOT / "code"))
from common import (  # noqa: E402
    DDRTREE_RANDOM_STATE,
    GENO_SHORT,
    SEED,
    assign_arms,
    dense,
)

CORES = int(os.environ.get("ANALYSIS_CORES", "8"))
BEAM_QVAL = 1e-4
trajectory_default = REPO_ROOT / "data" / "traj_ab.h5ad"
TRAJECTORY_INPUT = Path(
    os.environ.get("BIOBABEL_TRAJECTORY_INPUT", trajectory_default)
).expanduser().resolve()
OUTPUT_ROOT = Path(
    os.environ.get("BIOBABEL_OUTPUT_ROOT", REPO_ROOT / "figures")
).expanduser().resolve()

if not TRAJECTORY_INPUT.is_file():
    raise FileNotFoundError(
        f"Missing agent trajectory checkpoint: {TRAJECTORY_INPUT}. "
        "Deposit it at data/traj_ab.h5ad or set "
        "BIOBABEL_TRAJECTORY_INPUT."
    )

np.random.seed(SEED)
random.seed(SEED)
sub = ad.read_h5ad(TRAJECTORY_INPUT)
lineage_info = assign_arms(sub)
BP = lineage_info["branch_point"]
if BP is None:
    raise RuntimeError("Could not recover the frozen alpha/beta branch point")

beam_res = pd.read_csv(TABLES / "beam.csv.gz", index_col=0)
fig2g_values = pd.read_csv(TABLES / "fig2g.tsv", sep="\t")


def show(path):
    display(Image(filename=str(path)))


print(f"Trajectory checkpoint: {TRAJECTORY_INPUT}")
print(f"Cells: {sub.n_obs:,}; genes: {sub.n_vars:,}; branch point: {BP}")
print("No trajectory or expression model was refitted in this notebook.")


## Human-reviewed publication analysis

The following analyses reorganize the frozen agent results for the manuscript,
use the completed genotype screen across the predeclared 1,116-gene universe and
derive embryo-aware summaries. Kcnip4 is retained only in the Supplementary
cross-lineage screen.


In [ ]:
# Final publication setup and immutable provenance.
from collections import OrderedDict
from pathlib import Path
import hashlib
import subprocess
import igraph as ig

ROOT = REPO_ROOT
PUBLICATION_ROOT = OUTPUT_ROOT
FIG2_DIR = PUBLICATION_ROOT / "Fig2"
SFIG6_DIR = PUBLICATION_ROOT / "SFig6"
PUBLICATION_DATA = OUTPUT_ROOT / "source_data"
for directory in (FIG2_DIR, SFIG6_DIR, PUBLICATION_DATA):
    directory.mkdir(parents=True, exist_ok=True)

PUB_DPI = 600
FINAL_MARKERS = ["Neurog3", "Fev", "Gcg", "Ins2"]
GENOTYPE_ORDER = ["control", "dEndo", "DKO"]
GENOTYPE_COLOURS = {
    "control": "#0072B2",
    "dEndo": "#E69F00",
    "DKO": "#D55E00",
}
SCREEN_DIR = (
    ROOT / "tables"
)

def save_final_plot(plot, directory, stem, width, height):
    """Use ggplot2-python to export only PNG and PDF."""
    paths = []
    for extension in ("png", "pdf"):
        destination = directory / f"{stem}.{extension}"
        gg.ggsave(
            str(destination),
            plot=plot,
            width=width,
            height=height,
            dpi=PUB_DPI,
            bg="white",
        )
        paths.append(destination)
    return paths

def save_final_pheatmap(plot, directory, stem, width, height):
    """Use grid_py/Cairo to export a pheatmap as PNG and PDF."""
    grob = plot.gtable if hasattr(plot, "gtable") else plot
    paths = []
    for extension, surface_type in (("png", "image"), ("pdf", "pdf")):
        destination = directory / f"{stem}.{extension}"
        renderer = CairoRenderer(
            width=width,
            height=height,
            dpi=PUB_DPI,
            surface_type=surface_type,
            filename=str(destination) if surface_type != "image" else None,
            bg="white",
        )
        state = grid_py.get_state()
        state.reset()
        state.init_device(renderer)
        grid_py.grid_draw(grob)
        if surface_type == "image":
            renderer.write_to_png(str(destination))
        else:
            renderer.finish()
        paths.append(destination)
    return paths

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def git_commit(path):
    return subprocess.check_output(
        ["git", "-C", str(path), "rev-parse", "HEAD"],
        text=True,
    ).strip()

def expression_from_counts(adata, genes):
    """Per-cell log1p(CP10K) from the immutable raw-count layer."""
    symbols = adata.var["gene_short_name"].astype(str).to_numpy()
    index = {}
    for gene in genes:
        hits = np.flatnonzero(symbols == gene)
        if len(hits) != 1:
            raise ValueError(
                f"Expected one feature for {gene}; found {len(hits)}"
            )
        index[gene] = int(hits[0])
    selected = adata.layers["counts"][:, [index[g] for g in genes]]
    selected = dense(selected).astype(float)
    library = np.asarray(
        adata.layers["counts"].sum(axis=1)
    ).ravel().astype(float)
    library[library == 0] = 1
    return np.log1p(selected / library[:, None] * 1e4)

# Strict root-to-tip regions; side-path cells are not forced into alpha/beta.
pub_dd = sub.uns["monocle2"]["ddrtree"]
pub_centroids = np.asarray(pub_dd["K"]).T
pub_edges = np.asarray(pub_dd["mst_edges"], dtype=int)
pub_closest = np.asarray(
    pub_dd["closest_vertex"]
).ravel().astype(int)
pub_graph = ig.Graph(
    n=pub_centroids.shape[0],
    edges=[tuple(map(int, edge)) for edge in pub_edges],
)
pub_leaves = np.flatnonzero(
    np.asarray(pub_graph.degree()) == 1
)
pub_pseudotime = sub.obs["Pseudotime"].to_numpy(float)
pub_tip_expression = expression_from_counts(sub, ["Gcg", "Ins2"])
pub_centroid_summary = (
    pd.DataFrame(
        {
            "centroid": pub_closest,
            "pseudotime": pub_pseudotime,
            "Gcg": pub_tip_expression[:, 0],
            "Ins2": pub_tip_expression[:, 1],
        }
    )
    .groupby("centroid", observed=True)
    .mean()
)
pub_root = int(
    min(
        pub_leaves,
        key=lambda centroid: pub_centroid_summary.loc[
            centroid, "pseudotime"
        ]
        if centroid in pub_centroid_summary.index
        else np.inf,
    )
)
pub_terminal = [
    int(leaf) for leaf in pub_leaves if int(leaf) != pub_root
]
pub_alpha = int(
    max(
        pub_terminal,
        key=lambda centroid: pub_centroid_summary.loc[
            centroid, "Gcg"
        ]
        if centroid in pub_centroid_summary.index
        else -np.inf,
    )
)
pub_beta = int(
    max(
        pub_terminal,
        key=lambda centroid: pub_centroid_summary.loc[
            centroid, "Ins2"
        ]
        if centroid in pub_centroid_summary.index
        else -np.inf,
    )
)
pub_alpha_path = set(
    pub_graph.get_shortest_paths(pub_root, to=pub_alpha)[0]
)
pub_beta_path = set(
    pub_graph.get_shortest_paths(pub_root, to=pub_beta)[0]
)
pub_region = np.asarray(
    [
        "Shared trunk"
        if vertex in pub_alpha_path and vertex in pub_beta_path
        else "Alpha path"
        if vertex in pub_alpha_path
        else "Beta path"
        if vertex in pub_beta_path
        else "Side path"
        for vertex in pub_closest
    ]
)
sub.obs["publication_graph_region"] = pd.Categorical(
    pub_region,
    categories=[
        "Shared trunk", "Alpha path", "Beta path", "Side path"
    ],
    ordered=True,
)
sub.obs["geno_short"] = pd.Categorical(
    sub.obs["genotype"].map(GENO_SHORT),
    categories=GENOTYPE_ORDER,
    ordered=True,
)
print(pd.Series(pub_region).value_counts().to_string())


### Figure 2

The main demonstration contains the manually annotated DDRTree, four native
tradeSeq smoothers, the BEAM top-100 heatmap with selected lineage factors and
terminal-output genes labelled, and the comparison of fixed lineage-factor
and terminal-output programs.


In [ ]:
# Fig. 2f — manually annotated DDRTree.
publication_annotation_map = {
    "Neurog3-high endocrine progenitor": "Neurog3-high progenitor",
    "Fev+ endocrine precursor": "Fev+ precursor",
    "endocrine intermediate": "Alpha-route intermediate",
    "alpha-like endocrine": "Alpha-like",
    "Pdx1+ beta-like endocrine": "Beta-like",
    "mature beta-like endocrine": "Mature beta-like",
}
publication_annotation_colours = {
    "Neurog3-high progenitor": "#6A3D9A",
    "Fev+ precursor": "#56B4E9",
    "Alpha-route intermediate": "#E69F00",
    "Alpha-like": "#D55E00",
    "Beta-like": "#0072B2",
    "Mature beta-like": "#009E73",
}
sub.obs["publication_annotation"] = pd.Categorical(
    sub.obs["broad_annotation"].map(publication_annotation_map),
    categories=list(publication_annotation_colours),
    ordered=True,
)
fig2f = (
    m2.plot_cell_trajectory(
        sub,
        color_by="publication_annotation",
        cell_size=0.78,
        cell_link_size=0.28,
        backbone_color="#595959",
        show_branch_points=False,
    )
    + gg.scale_colour_manual(values=publication_annotation_colours)
    + gg.coord_fixed()
    + gg.labs(
        title=None, subtitle=None,
        x="DDRTree 1", y="DDRTree 2", colour=None,
    )
    + gg.theme_classic(base_size=9)
    + gg.theme(
        plot_title=gg.element_blank(),
        plot_subtitle=gg.element_blank(),
        axis_title=gg.element_text(size=8),
        axis_text=gg.element_text(size=7, colour="#555555"),
        axis_line=gg.element_line(
            colour="#555555", linewidth=0.35
        ),
        axis_ticks=gg.element_line(
            colour="#555555", linewidth=0.35
        ),
        legend_position="bottom",
        legend_text=gg.element_text(size=7),
        legend_key_width=gg.unit(0.30, "cm"),
        legend_key_height=gg.unit(0.30, "cm"),
    )
    + gg.guides(
        colour=gg.guide_legend(
            nrow=2,
            byrow=True,
            override_aes={"size": 2.2, "alpha": 1},
        )
    )
)
save_final_plot(
    fig2f, FIG2_DIR, "Fig2f_ddrtree_annotation", 6.7, 5.4
)
show(str(FIG2_DIR / "Fig2f_ddrtree_annotation.png"))

# Fig. 2g — reconstruct the original per-cell display from the frozen counts,
# and overlay the agent-fitted tradeSeq smoother values. This is plotting only:
# no GAM is fitted here. Shared-trunk cells receive a deterministic display
# lineage; that assignment changes point colour only, not the frozen curves.
fig2g_weights = np.asarray(sub.obsm["cell_weights"], dtype=float)
fig2g_prob = fig2g_weights / fig2g_weights.sum(axis=1, keepdims=True)
fig2g_rng = np.random.default_rng(SEED)
fig2g_assignment = np.where(
    fig2g_rng.random(sub.n_obs) < fig2g_prob[:, 0],
    "Alpha",
    "Beta",
)
fig2g_symbols = sub.var["gene_short_name"].astype(str).to_numpy()
fig2g_gene_index = {}
for marker in FINAL_MARKERS:
    hits = np.flatnonzero(fig2g_symbols == marker)
    if len(hits) != 1:
        raise ValueError(
            f"Expected one feature for {marker}; found {len(hits)}"
        )
    fig2g_gene_index[marker] = int(hits[0])
fig2g_counts = dense(
    sub.layers["counts"][:, [fig2g_gene_index[g] for g in FINAL_MARKERS]]
).astype(float)
fig2g_points = pd.concat(
    [
        pd.DataFrame(
            {
                "time": sub.obs["Pseudotime"].to_numpy(float),
                "log1p_count": np.log1p(fig2g_counts[:, index]),
                "lineage_name": fig2g_assignment,
                "gene": marker,
            }
        )
        for index, marker in enumerate(FINAL_MARKERS)
    ],
    ignore_index=True,
)
fig2g_gene_order = ["Fev", "Gcg", "Ins2", "Neurog3"]
fig2g_points["lineage_name"] = pd.Categorical(
    fig2g_points["lineage_name"],
    categories=["Alpha", "Beta"],
    ordered=True,
)
fig2g_points["gene"] = pd.Categorical(
    fig2g_points["gene"],
    categories=fig2g_gene_order,
    ordered=True,
)
fig2g_values["lineage_name"] = pd.Categorical(
    fig2g_values["lineage_name"],
    categories=["Alpha", "Beta"],
    ordered=True,
)
fig2g_values["gene"] = pd.Categorical(
    fig2g_values["gene"],
    categories=fig2g_gene_order,
    ordered=True,
)
fig2g_values["log1p_yhat"] = np.log1p(fig2g_values["yhat"])
fig2g = (
    gg.ggplot(
        fig2g_points,
        gg.aes(x="time", y="log1p_count", colour="lineage_name"),
    )
    + gg.geom_point(size=0.18, alpha=0.20)
    + gg.facet_wrap("gene", ncol=2)
    + gg.scale_colour_manual(
        values={"Alpha": "#D55E00", "Beta": "#0072B2"},
        breaks=["Alpha", "Beta"],
    )
    + gg.labs(
        title=None,
        subtitle=None,
        x="Pseudotime",
        y="Log(expression + 1)",
        colour=None,
    )
    + gg.theme_classic(base_size=9)
    + gg.theme(
        axis_title=gg.element_text(size=8),
        axis_text=gg.element_text(size=7, colour="#4D4D4D"),
        strip_text=gg.element_text(size=9, face="italic"),
        legend_position="bottom",
        legend_text=gg.element_text(size=8),
        legend_key_width=gg.unit(0.55, "cm"),
    )
    + gg.guides(
        colour=gg.guide_legend(
            nrow=1, override_aes={"size": 1.8, "alpha": 1}
        )
    )
)
for fig2g_lineage, fig2g_colour in (
    ("Alpha", "#D55E00"),
    ("Beta", "#0072B2"),
):
    fig2g_line = fig2g_values.loc[
        fig2g_values["lineage_name"] == fig2g_lineage
    ].copy()
    fig2g = (
        fig2g
        + gg.geom_line(
            data=fig2g_line,
            mapping=gg.aes(
                x="time", y="log1p_yhat", group="gene"
            ),
            inherit_aes=False,
            colour="white",
            linewidth=1.85,
        )
        + gg.geom_line(
            data=fig2g_line,
            mapping=gg.aes(
                x="time", y="log1p_yhat", group="gene"
            ),
            inherit_aes=False,
            colour=fig2g_colour,
            linewidth=0.85,
        )
    )
save_final_plot(
    fig2g, FIG2_DIR, "Fig2g_tradeseq_four_genes", 6.2, 4.8
)
show(FIG2_DIR / "Fig2g_tradeseq_four_genes.png")


In [ ]:
# Fig. 2h — original BEAM top 100 with selected biologically relevant labels.
FINAL_BEAM_LABELS = {
    "Arx", "Irx1", "Irx2", "Pou3f4", "Gcg", "Ttr",
    "Pdx1", "Nkx6-1", "Mafa", "Ins1", "Ins2", "Iapp",
}
beam_final = beam_res.sort_values("qval", kind="mergesort")
beam_top100_ids = [
    feature
    for feature in beam_final.index[
        beam_final["qval"] < BEAM_QVAL
    ]
    if feature in sub.var_names
][:100]
if len(beam_top100_ids) != 100:
    raise RuntimeError(
        f"Expected 100 BEAM genes; found {len(beam_top100_ids)}"
    )
beam_native = m2.plot_genes_branched_heatmap(
    sub[:, beam_top100_ids],
    branch_point=BP,
    num_clusters=4,
    branch_labels=("alpha", "beta"),
    use_gene_short_name=True,
    show_rownames=False,
    cluster_rows=True,
    return_heatmap=True,
    cores=CORES,
)
beam_matrix = beam_native["heatmap_matrix"].copy()
beam_symbols = (
    sub.var.loc[beam_matrix.index, "gene_short_name"]
    .astype(str)
    .tolist()
)
missing_beam_labels = sorted(
    FINAL_BEAM_LABELS.difference(beam_symbols)
)
if missing_beam_labels:
    raise RuntimeError(
        "Selected labels absent from BEAM top 100: "
        + ", ".join(missing_beam_labels)
    )
beam_display_labels = [
    symbol if symbol in FINAL_BEAM_LABELS else "\u200b" * (i + 1)
    for i, symbol in enumerate(beam_symbols)
]
beam_annotation_row = beam_native["annotation_row"].copy()
beam_annotation_row.index = beam_display_labels
beam_matrix.columns = [
    str(value) for value in beam_matrix.columns
]
beam_min = float(beam_matrix.to_numpy().min())
beam_max = float(beam_matrix.to_numpy().max())
beam_breaks = np.arange(
    beam_min - 0.1, beam_max + 0.1 + 1e-9, 0.1
)
fig2h = pheatmap.pheatmap(
    beam_matrix,
    cluster_cols=False,
    cluster_rows=beam_native["ph"].tree_row,
    cutree_rows=4,
    show_colnames=False,
    show_rownames=True,
    labels_row=beam_display_labels,
    annotation_row=beam_annotation_row,
    annotation_col=beam_native["annotation_col"],
    annotation_colors=beam_native["annotation_colors"],
    annotation_names_row=False,
    annotation_names_col=False,
    gaps_col=[beam_native["col_gap_ind"]],
    treeheight_row=20,
    breaks=beam_breaks,
    color=beam_native["hmcols"],
    border_color=None,
    fontsize=7,
    fontsize_row=7,
    main=None,
    silent=True,
)
save_final_pheatmap(
    fig2h, FIG2_DIR, "Fig2h_BEAM_top100", 6.4, 7.4
)
beam_top100_table = beam_final.loc[beam_top100_ids].copy()
beam_top100_table.insert(
    0, "beam_rank", np.arange(1, len(beam_top100_table) + 1)
)
beam_top100_table["labelled"] = (
    beam_top100_table["gene_short_name"].astype(str)
    .isin(FINAL_BEAM_LABELS)
)
beam_top100_table.to_csv(
    PUBLICATION_DATA / "fig2h.tsv",
    sep="\t",
    index=True,
)
show(str(FIG2_DIR / "Fig2h_BEAM_top100.png"))


In [ ]:
# Fig. 2i — fixed, biologically curated lineage-TF and terminal-output programs.
# These publication-stage gene sets are not an unsupervised genotype-effect
# selection; genotype effects are tested separately below.
FINAL_PROGRAMS = OrderedDict(
    [
        (
            "α lineage TFs",
            {
                "lineage": "Alpha",
                "genes": ["Arx", "Irx1", "Irx2", "Pou3f4"],
            },
        ),
        (
            "α terminal output",
            {
                "lineage": "Alpha",
                "genes": ["Gcg", "Ttr"],
            },
        ),
        (
            "β lineage TFs",
            {
                "lineage": "Beta",
                "genes": ["Pdx1", "Nkx6-1", "Mafa"],
            },
        ),
        (
            "β terminal output",
            {
                "lineage": "Beta",
                "genes": ["Ins1", "Ins2", "Iapp"],
            },
        ),
    ]
)
# Selected Fig. 2i genes and concise biological rationale.
gene_selection_records = [
    (
        "Arx", "alpha lineage TF",
        "Alpha-cell fate specification and identity maintenance.",
        "Canonical alpha-lineage regulator and BEAM top-100 gene.",
    ),
    (
        "Irx1", "alpha lineage TF",
        "Embryonic glucagon-positive-cell-enriched transcription factor.",
        "Alpha-identity marker and BEAM top-100 gene.",
    ),
    (
        "Irx2", "alpha lineage TF",
        "Embryonic glucagon-positive-cell-enriched transcription factor.",
        "Alpha-identity marker and BEAM top-100 gene.",
    ),
    (
        "Pou3f4", "alpha lineage TF",
        "Alpha-associated transcription factor that promotes Gcg expression.",
        "Alpha-identity marker and BEAM top-100 gene.",
    ),
    (
        "Gcg", "alpha terminal output",
        "Defining alpha-cell hormone output.",
        "Strongest alpha genotype effect and BEAM top-100 gene.",
    ),
    (
        "Ttr", "alpha output",
        "Alpha-associated secreted transcript.",
        "Independent alpha-output marker and BEAM top-100 gene.",
    ),
    (
        "Pdx1", "beta lineage TF",
        "Pancreatic development and beta-cell identity maintenance.",
        "Canonical beta-lineage regulator and BEAM top-100 gene.",
    ),
    (
        "Nkx6-1", "beta lineage TF",
        "Beta-cell fate establishment and identity maintenance.",
        "Canonical beta-lineage regulator and BEAM top-100 gene.",
    ),
    (
        "Mafa", "beta lineage TF",
        "Insulin transcription and beta-cell functional maturation.",
        "Canonical beta-maturation regulator and BEAM top-100 gene.",
    ),
    (
        "Ins1", "beta terminal output",
        "Core beta-cell insulin output.",
        "Strongest beta genotype effect and BEAM top-100 gene.",
    ),
    (
        "Ins2", "beta terminal output",
        "Core beta-cell insulin output.",
        "Canonical beta-output and BEAM top-100 gene.",
    ),
    (
        "Iapp", "beta terminal output",
        "Beta-cell amylin secretory output.",
        "Independent beta-output and top genotype-effect gene.",
    ),
]
gene_selection_table = pd.DataFrame(
    gene_selection_records,
    columns=[
        "gene", "group", "function", "selection_rationale",
    ],
)
beam_selection_meta = (
    beam_top100_table.reset_index(drop=False)
    .set_index("gene_short_name")
)
gene_selection_table["beam_top100_rank"] = (
    gene_selection_table["gene"]
    .map(beam_selection_meta["beam_rank"])
    .astype("Int64")
)
gene_selection_table.to_csv(
    PUBLICATION_DATA / "fig2i_genes.tsv",
    sep="\t",
    index=False,
)
display(gene_selection_table)

program_genes = list(
    dict.fromkeys(
        gene
        for definition in FINAL_PROGRAMS.values()
        for gene in definition["genes"]
    )
)
program_expression = expression_from_counts(sub, program_genes)
program_index = {
    gene: index for index, gene in enumerate(program_genes)
}
program_rows = []
program_bin_rows = []
for lineage in ("Alpha", "Beta"):
    terminal_region = f"{lineage} path"
    lineage_mask = np.isin(
        pub_region, ["Shared trunk", terminal_region]
    )
    control_mask = (
        lineage_mask
        & sub.obs["geno_short"].astype(str).eq("control").to_numpy()
    )
    edges = np.quantile(
        pub_pseudotime[control_mask], np.linspace(0, 1, 6)
    )
    edges[0] -= 1e-8
    edges[-1] += 1e-8
    bin_id = np.digitize(pub_pseudotime, edges[1:-1]) + 1
    for edge_index, edge in enumerate(edges):
        program_bin_rows.append(
            {
                "lineage": lineage,
                "edge_index": edge_index,
                "pseudotime_edge": float(edge),
                "definition": "control-cell quantiles",
            }
        )
    lineage_programs = {
        panel: definition["genes"]
        for panel, definition in FINAL_PROGRAMS.items()
        if definition["lineage"] == lineage
    }
    for sample in sub.obs.loc[lineage_mask, "sample"].unique():
        sample_mask = (
            lineage_mask
            & sub.obs["sample"].eq(sample).to_numpy()
        )
        genotype = str(
            sub.obs.loc[sample_mask, "geno_short"].iloc[0]
        )
        for bin_number in range(1, 6):
            selected = sample_mask & (bin_id == bin_number)
            if not selected.any():
                continue
            for panel, genes in lineage_programs.items():
                gene_means = [
                    float(
                        program_expression[
                            selected, program_index[gene]
                        ].mean()
                    )
                    for gene in genes
                ]
                program_rows.append(
                    {
                        "lineage": lineage,
                        "sample": str(sample),
                        "genotype": genotype,
                        "pseudotime_bin": bin_number,
                        "panel": panel,
                        "program_log1pCP10K": float(
                            np.mean(gene_means)
                        ),
                        "n_cells": int(selected.sum()),
                    }
                )

program_table = pd.DataFrame(program_rows)
program_table["genotype"] = pd.Categorical(
    program_table["genotype"],
    categories=GENOTYPE_ORDER,
    ordered=True,
)
program_table["panel"] = pd.Categorical(
    program_table["panel"],
    categories=list(FINAL_PROGRAMS),
    ordered=True,
)
control_reference = (
    program_table[
        program_table["genotype"].astype(str) == "control"
    ]
    .groupby(
        ["panel", "pseudotime_bin"], observed=True
    )["program_log1pCP10K"]
    .mean()
    .rename("control_reference")
)
program_table = program_table.join(
    control_reference,
    on=["panel", "pseudotime_bin"],
)
program_table["delta_vs_control"] = (
    program_table["program_log1pCP10K"]
    - program_table["control_reference"]
)
program_summary = (
    program_table.groupby(
        ["panel", "genotype", "pseudotime_bin"],
        observed=True,
    )
    .agg(
        mean=("program_log1pCP10K", "mean"),
        sd=("program_log1pCP10K", "std"),
        n_embryos=("sample", "nunique"),
    )
    .reset_index()
)
program_summary["se"] = (
    program_summary["sd"]
    / np.sqrt(program_summary["n_embryos"])
)
program_summary["ymin"] = (
    np.maximum(
        0.0,
        program_summary["mean"] - program_summary["se"],
    )
)
program_summary["ymax"] = (
    program_summary["mean"] + program_summary["se"]
)
program_offsets = {
    "control": -0.06, "dEndo": 0.0, "DKO": 0.06
}
program_summary["plot_x"] = (
    program_summary["pseudotime_bin"].astype(float)
    + program_summary["genotype"].astype(str).map(program_offsets)
)
program_summary["cap_left"] = program_summary["plot_x"] - 0.05
program_summary["cap_right"] = program_summary["plot_x"] + 0.05

# Predeclared program-level tests reuse the completed sex-adjusted
# tradeSeq member-gene tests. Within each program and contrast, raw
# effect-threshold p-values are combined by Simes; BH correction is then
# applied across the eight program-by-contrast hypotheses.
def program_simes_pvalue(pvalues):
    ordered = np.sort(np.asarray(pvalues, dtype=float))
    ranks = np.arange(1, len(ordered) + 1)
    return float(
        min(1.0, np.min(ordered * len(ordered) / ranks))
    )

def program_bh_adjust(pvalues):
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values, kind="mergesort")
    adjusted = np.empty(len(values), dtype=float)
    running = 1.0
    for position in range(len(values) - 1, -1, -1):
        index = order[position]
        running = min(
            running,
            values[index] * len(values) / (position + 1),
        )
        adjusted[index] = running
    return adjusted

def program_q_stars(qvalue):
    if qvalue < 0.001:
        return "***"
    if qvalue < 0.01:
        return "**"
    if qvalue < 0.05:
        return "*"
    return "ns"

program_gene_tests = pd.read_csv(
    SCREEN_DIR / "screen_control.tsv.gz",
    sep="\t",
)
program_test_rows = []
for panel, definition in FINAL_PROGRAMS.items():
    lineage_key = definition["lineage"].lower()
    for contrast in ("dEndo-control", "DKO-control"):
        context = program_gene_tests[
            program_gene_tests["lineage"].astype(str).eq(
                lineage_key
            )
            & program_gene_tests["contrast"].astype(str).eq(
                contrast
            )
            & program_gene_tests["gene_symbol"].isin(
                definition["genes"]
            )
        ].copy()
        context = context.set_index("gene_symbol").loc[
            definition["genes"]
        ].reset_index()
        if len(context) != len(definition["genes"]):
            raise RuntimeError(
                f"Missing member-gene test for {panel}, {contrast}"
            )
        member_pvalues = context[
            "pvalue_l2fc_0_5"
        ].to_numpy(float)
        program_test_rows.append(
            {
                "panel": panel,
                "lineage": lineage_key,
                "contrast": contrast,
                "genes": ",".join(definition["genes"]),
                "n_genes": len(definition["genes"]),
                "member_pvalues_l2fc_0_5": ";".join(
                    f"{gene}={pvalue:.8g}"
                    for gene, pvalue in zip(
                        context["gene_symbol"],
                        member_pvalues,
                    )
                ),
                "simes_pvalue_l2fc_0_5": (
                    program_simes_pvalue(member_pvalues)
                ),
            }
        )
program_test_table = pd.DataFrame(program_test_rows)
program_test_table["BH_qvalue_l2fc_0_5"] = (
    program_bh_adjust(
        program_test_table[
            "simes_pvalue_l2fc_0_5"
        ].to_numpy(float)
    )
)
program_test_table["stars"] = program_test_table[
    "BH_qvalue_l2fc_0_5"
].map(program_q_stars)
program_test_table["test_definition"] = (
    "Simes combination of member-gene sex-adjusted tradeSeq "
    "condition-test p-values for |log2FC| > 0.5; BH across "
    "8 fixed program-by-contrast hypotheses"
)

program_test_bars = []
program_test_connectors = []
program_terminal = (
    program_summary[
        program_summary["pseudotime_bin"] == 5
    ]
    .set_index(["panel", "genotype"])
)
for panel in FINAL_PROGRAMS:
    control_endpoint = program_terminal.loc[
        (panel, "control")
    ]
    for row_index, (contrast, mutant) in enumerate(
        [
            ("dEndo-control", "dEndo"),
            ("DKO-control", "DKO"),
        ]
    ):
        mutant_endpoint = program_terminal.loc[
            (panel, mutant)
        ]
        test_row = program_test_table[
            program_test_table["panel"].eq(panel)
            & program_test_table["contrast"].eq(contrast)
        ].iloc[0]
        bar_x = 5.20 + row_index * 0.40
        control_y = float(control_endpoint["mean"])
        mutant_y = float(mutant_endpoint["mean"])
        program_test_bars.append(
            {
                "panel": panel,
                "bar_x": bar_x,
                "control_y": control_y,
                "mutant_y": mutant_y,
                "star_x": bar_x + 0.10,
                "star_y": (
                    control_y + mutant_y
                ) / 2.0,
                "stars": str(test_row["stars"]),
            }
        )
        program_test_connectors.extend(
            [
                {
                    "panel": panel,
                    "x_start": bar_x - 0.07,
                    "x_end": bar_x,
                    "connector_y": control_y,
                },
                {
                    "panel": panel,
                    "x_start": bar_x - 0.07,
                    "x_end": bar_x,
                    "connector_y": mutant_y,
                },
            ]
        )
program_test_bars = pd.DataFrame(program_test_bars)
program_test_connectors = pd.DataFrame(
    program_test_connectors
)
for annotation_table in (
    program_test_bars,
    program_test_connectors,
):
    annotation_table["panel"] = pd.Categorical(
        annotation_table["panel"],
        categories=list(FINAL_PROGRAMS),
        ordered=True,
    )

program_zero = pd.DataFrame(
    {
        "panel": pd.Categorical(
            list(FINAL_PROGRAMS),
            categories=list(FINAL_PROGRAMS),
            ordered=True,
        ),
        "plot_x": [1.0] * len(FINAL_PROGRAMS),
        "mean": [0.0] * len(FINAL_PROGRAMS),
    }
)
fig2i = (
    gg.ggplot()
    + gg.geom_blank(
        data=program_zero,
        mapping=gg.aes(x="plot_x", y="mean"),
        inherit_aes=False,
    )
    + gg.facet_wrap("panel", ncol=2, scales="free_y")
    + gg.scale_colour_manual(
        values=GENOTYPE_COLOURS,
        breaks=GENOTYPE_ORDER,
    )
    + gg.scale_x_continuous(
        breaks=[1, 2, 3, 4, 5],
        limits=(0.72, 6.02),
        expand=gg.expansion(mult=(0, 0)),
    )
    + gg.scale_y_continuous(
        expand=gg.expansion(mult=(0.04, 0.10)),
    )
    + gg.labs(
        title=None,
        subtitle=None,
        x="Pseudotime bin",
        y="Mean expression\n(log1p(CP10K))",
        colour=None,
    )
    + gg.theme_minimal(base_size=9)
    + gg.theme(
        plot_title=gg.element_blank(),
        plot_subtitle=gg.element_blank(),
        axis_title=gg.element_text(size=8),
        axis_text=gg.element_text(size=7, colour="#555555"),
        panel_grid_minor=gg.element_blank(),
        panel_grid_major_x=gg.element_blank(),
        panel_grid_major_y=gg.element_line(
            colour="#E7E7E7",
            linewidth=0.25,
        ),
        strip_background=gg.element_blank(),
        strip_text=gg.element_text(size=8, face="bold"),
        panel_spacing=gg.unit(0.28, "in"),
        legend_position="top",
        legend_direction="horizontal",
        legend_title=gg.element_blank(),
        legend_text=gg.element_text(size=7),
    )
)
# Build uncertainty ribbons and mean trajectories as fixed-colour layers.
for genotype, colour in GENOTYPE_COLOURS.items():
    genotype_summary = program_summary[
        program_summary["genotype"].astype(str) == genotype
    ].copy()
    genotype_summary["genotype"] = genotype
    fig2i = (
        fig2i
        + gg.geom_ribbon(
            data=genotype_summary,
            mapping=gg.aes(
                x="pseudotime_bin",
                ymin="ymin",
                ymax="ymax",
            ),
            inherit_aes=False,
            fill=colour,
            colour=None,
            alpha=0.15,
        )
        + gg.geom_line(
            data=genotype_summary,
            mapping=gg.aes(
                x="pseudotime_bin",
                y="mean",
            ),
            inherit_aes=False,
            colour=colour,
            linewidth=1.0,
        )
    )
fig2i = (
    fig2i
    + gg.geom_point(
        data=program_summary,
        mapping=gg.aes(
            x="pseudotime_bin",
            y="mean",
            colour="genotype",
        ),
        inherit_aes=False,
        size=1.35,
    )
    + gg.geom_segment(
        data=program_test_connectors,
        mapping=gg.aes(
            x="x_start",
            xend="x_end",
            y="connector_y",
            yend="connector_y",
        ),
        inherit_aes=False,
        colour="#5A5A5A",
        linewidth=0.4,
    )
    + gg.geom_segment(
        data=program_test_bars,
        mapping=gg.aes(
            x="bar_x",
            xend="bar_x",
            y="control_y",
            yend="mutant_y",
        ),
        inherit_aes=False,
        colour="#5A5A5A",
        linewidth=0.4,
    )
    + gg.geom_text(
        data=program_test_bars,
        mapping=gg.aes(
            x="star_x",
            y="star_y",
            label="stars",
        ),
        inherit_aes=False,
        colour="#333333",
        size=2.25,
        hjust=0,
        vjust=0.5,
    )
)
save_final_plot(
    fig2i,
    FIG2_DIR,
    "Fig2i_identity_terminal_output_programs",
    6.8,
    4.8,
)
program_table.to_csv(
    PUBLICATION_DATA / "fig2i_values.tsv",
    sep="\t",
    index=False,
)
pd.DataFrame(program_bin_rows).to_csv(
    PUBLICATION_DATA / "fig2i_bins.tsv",
    sep="\t",
    index=False,
)
program_test_table.to_csv(
    PUBLICATION_DATA / "fig2i_tests.tsv",
    sep="\t",
    index=False,
)
show(
    str(
        FIG2_DIR /
        "Fig2i_identity_terminal_output_programs.png"
    )
)


### Supplementary Figure 6

The trajectory panels reuse the frozen DDRTree. Allocation summaries use
embryos as the experimental units. The cross-lineage display uses the frozen
completed genotype-screen statistics and does not refit tradeSeq.


In [ ]:
# SFig. 6a-f and i — markers, genotype, pseudotime and state on the frozen DDRTree.
def trajectory_publication_theme(plot, legend_position="right"):
    return (
        plot
        + gg.coord_fixed()
        + gg.labs(
            title=None, subtitle=None,
            x="DDRTree 1", y="DDRTree 2",
        )
        + gg.theme_classic(base_size=9)
        + gg.theme(
            plot_title=gg.element_blank(),
            plot_subtitle=gg.element_blank(),
            axis_title=gg.element_text(size=8),
            axis_text=gg.element_text(size=7, colour="#555555"),
            axis_line=gg.element_line(
                colour="#555555", linewidth=0.35
            ),
            axis_ticks=gg.element_line(
                colour="#555555", linewidth=0.35
            ),
            legend_position=legend_position,
            legend_title=gg.element_text(size=8),
            legend_text=gg.element_text(size=7),
            legend_key_width=gg.unit(0.30, "cm"),
            legend_key_height=gg.unit(0.30, "cm"),
        )
    )

for panel_letter, marker in zip("abcd", FINAL_MARKERS):
    marker_key = f"publication_{marker}"
    sub.obs[marker_key] = expression_from_counts(
        sub, [marker]
    )[:, 0]
    marker_plot = (
        trajectory_publication_theme(
            m2.plot_cell_trajectory(
                sub,
                color_by=marker_key,
                cell_size=0.72,
                cell_link_size=0.28,
                backbone_color="#595959",
                show_branch_points=False,
            ),
            legend_position="right",
        )
        + gg.scale_colour_viridis_c(name=marker)
        + gg.guides(
            colour=gg.guide_colorbar(direction="vertical")
        )
    )
    stem = f"SFig6{panel_letter}_ddrtree_{marker}"
    save_final_plot(
        marker_plot, SFIG6_DIR, stem, 3.6, 3.1
    )
    show(str(SFIG6_DIR / f"{stem}.png"))

GENOTYPE_TRAJECTORY_COLOURS = {
    "control": "#0072B2",
    "dEndo": "#E69F00",
    "DKO": "#D55E00",
}
genotype_plot = (
    trajectory_publication_theme(
        m2.plot_cell_trajectory(
            sub,
            color_by="geno_short",
            cell_size=0.62,
            cell_link_size=0.28,
            backbone_color="#595959",
            show_branch_points=False,
        ),
        legend_position="bottom",
    )
    + gg.scale_colour_manual(values=GENOTYPE_TRAJECTORY_COLOURS)
    + gg.labs(colour=None)
    + gg.guides(
        colour=gg.guide_legend(
            nrow=1,
            override_aes={"size": 2.2, "alpha": 1},
        )
    )
)
save_final_plot(
    genotype_plot,
    SFIG6_DIR,
    "SFig6e_ddrtree_genotype",
    4.4,
    3.4,
)
show(str(SFIG6_DIR / "SFig6e_ddrtree_genotype.png"))

pseudotime_plot = (
    trajectory_publication_theme(
        m2.plot_cell_trajectory(
            sub,
            color_by="Pseudotime",
            cell_size=0.78,
            cell_link_size=0.28,
            backbone_color="#595959",
            show_branch_points=False,
        ),
        legend_position="right",
    )
    + gg.labs(colour="Pseudotime")
)
save_final_plot(
    pseudotime_plot,
    SFIG6_DIR,
    "SFig6f_ddrtree_pseudotime",
    3.6,
    3.1,
)
show(str(SFIG6_DIR / "SFig6f_ddrtree_pseudotime.png"))

state_levels = [
    str(value)
    for value in sorted(
        pd.to_numeric(sub.obs["State"]).astype(int).unique()
    )
]
sub.obs["publication_state"] = pd.Categorical(
    pd.to_numeric(sub.obs["State"]).astype(int).astype(str),
    categories=state_levels,
    ordered=True,
)
STATE_TRAJECTORY_COLOURS = {
    "1": "#D55E00",
    "2": "#E69F00",
    "3": "#009E73",
    "4": "#0072B2",
    "5": "#CC79A7",
}
state_plot = (
    trajectory_publication_theme(
        m2.plot_cell_trajectory(
            sub,
            color_by="publication_state",
            cell_size=0.62,
            cell_link_size=0.45,
            backbone_color="#222222",
            show_state_number=False,
            show_branch_points=False,
        ),
        legend_position="bottom",
    )
    + gg.scale_colour_manual(
        values=STATE_TRAJECTORY_COLOURS,
        breaks=state_levels,
        name="State",
    )
    + gg.guides(
        colour=gg.guide_legend(
            nrow=1,
            override_aes={"size": 2.2, "alpha": 1},
        )
    )
)
save_final_plot(
    state_plot,
    SFIG6_DIR,
    "SFig6i_ddrtree_state",
    4.4,
    3.4,
)
show(str(SFIG6_DIR / "SFig6i_ddrtree_state.png"))


In [ ]:
# SFig. 6g — embryo-level allocation from the frozen summary table.
allocation = pd.read_csv(TABLES / "sfig6g.tsv", sep="\t")
allocation_tests = pd.read_csv(
    TABLES / "sfig6g_tests.tsv", sep="\t"
)
allocation_labels = OrderedDict(
    [
        ("beta_among_terminal", "β / (α + β)"),
        ("terminal_among_all", "(α + β) / all trajectory"),
    ]
)
allocation_long = allocation.melt(
    id_vars=["sample", "geno_short"],
    value_vars=list(allocation_labels),
    var_name="metric_key",
    value_name="fraction",
)
allocation_long["metric"] = pd.Categorical(
    allocation_long["metric_key"].map(allocation_labels),
    categories=list(allocation_labels.values()),
    ordered=True,
)
allocation_long["geno_short"] = pd.Categorical(
    allocation_long["geno_short"],
    categories=GENOTYPE_ORDER,
    ordered=True,
)
allocation_long = allocation_long.sort_values(
    ["metric", "geno_short", "sample"]
)
allocation_long["offset_index"] = allocation_long.groupby(
    ["metric", "geno_short"], observed=True
).cumcount()
allocation_long["offset_n"] = allocation_long.groupby(
    ["metric", "geno_short"], observed=True
)["sample"].transform("size")
allocation_long["x"] = (
    allocation_long["geno_short"].cat.codes.astype(float)
    + np.where(
        allocation_long["offset_n"] > 1,
        (allocation_long["offset_index"]
         - (allocation_long["offset_n"] - 1) / 2) * 0.10,
        0.0,
    )
)
allocation_medians = (
    allocation_long.groupby(
        ["metric", "geno_short"], observed=True
    )["fraction"]
    .median()
    .reset_index()
)
allocation_medians["x"] = (
    allocation_medians["geno_short"].cat.codes.astype(float)
)
allocation_medians["x_start"] = allocation_medians["x"] - 0.17
allocation_medians["x_end"] = allocation_medians["x"] + 0.17
allocation_test_labels = allocation_tests.rename(
    columns={"pvalue": "pvalue"}
).copy()
allocation_test_labels["metric"] = pd.Categorical(
    allocation_test_labels["metric"],
    categories=list(allocation_labels.values()),
    ordered=True,
)
allocation_test_labels["label"] = allocation_test_labels[
    "pvalue"
].map(lambda value: f"Kruskal–Wallis P = {value:.2f}")
allocation_test_labels["x"] = 1.0
allocation_metric_max = allocation_long.groupby(
    "metric", observed=True
)["fraction"].max()
allocation_test_labels["fraction"] = (
    allocation_test_labels["metric"].map(allocation_metric_max).astype(float)
    + 0.06
)

allocation_plot = (
    gg.ggplot(
        allocation_long,
        gg.aes(x="x", y="fraction", colour="geno_short"),
    )
    + gg.geom_point(size=2.0)
    + gg.geom_segment(
        data=allocation_medians,
        mapping=gg.aes(
            x="x_start", xend="x_end",
            y="fraction", yend="fraction",
            colour="geno_short",
        ),
        inherit_aes=False,
        linewidth=0.9,
    )
    + gg.geom_text(
        data=allocation_test_labels,
        mapping=gg.aes(x="x", y="fraction", label="label"),
        inherit_aes=False,
        size=2.5,
    )
    + gg.facet_wrap("metric", ncol=2, scales="free_y")
    + gg.scale_colour_manual(values=GENOTYPE_COLOURS)
    + gg.scale_x_continuous(
        breaks=[0, 1, 2], labels=GENOTYPE_ORDER
    )
    + gg.labs(x=None, y="Fraction", colour=None)
    + gg.theme_classic(base_size=9)
    + gg.theme(
        axis_title=gg.element_text(size=8),
        axis_text=gg.element_text(size=7, colour="#555555"),
        strip_background=gg.element_blank(),
        strip_text=gg.element_text(size=8, face="bold"),
        legend_position="none",
    )
)
save_final_plot(
    allocation_plot,
    SFIG6_DIR,
    "SFig6g_embryo_allocation",
    6.8,
    3.6,
)
show(SFIG6_DIR / "SFig6g_embryo_allocation.png")


In [ ]:
# SFig. 6h — effect-size-aware cross-lineage genotype screen.
#
# The plotted values are frozen from the completed 1,116-gene, sex-adjusted
# tradeSeq screen. The display universe additionally requires control mean
# log1p(CP10K) >= 0.1 in both trajectories. A shared category requires the
# same effect direction and BH q(|log2FC| > 0.5) < 0.05 in both alpha and beta.
screen_values_path = (
    TABLES / "sfig6h.tsv"
)
cross_lineage_screen = pd.read_csv(screen_values_path, sep="\t")

contrast_order = ["dEndo − control", "DKO − control"]
category_order = [
    "shared decrease",
    "shared increase",
]
category_labels = {
    "shared decrease (q < 0.05)": "shared decrease",
    "shared increase (q < 0.05)": "shared increase",
}
category_colours = {
    "shared decrease": "#4C92C3",
    "shared increase": "#E69F00",
}

cross_lineage_screen["contrast"] = pd.Categorical(
    cross_lineage_screen["contrast"],
    categories=contrast_order,
    ordered=True,
)
cross_lineage_screen["display_category"] = pd.Categorical(
    cross_lineage_screen["category"].map(category_labels),
    categories=category_order,
    ordered=True,
)

screen_other = cross_lineage_screen.loc[
    cross_lineage_screen["gene"].ne("Kcnip4")
    & cross_lineage_screen["display_category"].isna()
].copy()
screen_highlight = cross_lineage_screen.loc[
    cross_lineage_screen["gene"].ne("Kcnip4")
    & cross_lineage_screen["display_category"].notna()
].copy()
kcnip4_screen = cross_lineage_screen.loc[
    cross_lineage_screen["gene"] == "Kcnip4"
].copy()

screen_spans = {}
screen_bound_rows = []
for contrast, context in cross_lineage_screen.groupby(
    "contrast", observed=True
):
    lower = min(
        float(context["alpha_effect"].min()),
        float(context["beta_effect"].min()),
    )
    upper = max(
        float(context["alpha_effect"].max()),
        float(context["beta_effect"].max()),
    )
    screen_spans[str(contrast)] = upper - lower
    x_radius = max(
        abs(float(context["alpha_effect"].min())),
        abs(float(context["alpha_effect"].max())),
    ) * 1.04
    screen_bound_rows.extend(
        [
            {
                "contrast": contrast,
                "x_bound": -x_radius,
                "y_bound": 0.0,
            },
            {
                "contrast": contrast,
                "x_bound": x_radius,
                "y_bound": 0.0,
            },
        ]
    )
screen_x_bounds = pd.DataFrame(screen_bound_rows)
screen_x_bounds["contrast"] = pd.Categorical(
    screen_x_bounds["contrast"],
    categories=contrast_order,
    ordered=True,
)
kcnip4_screen["label_x"] = [
    effect + 0.025 * screen_spans[str(contrast)]
    for effect, contrast in zip(
        kcnip4_screen["alpha_effect"],
        kcnip4_screen["contrast"],
    )
]
kcnip4_screen["label_y"] = [
    effect + 0.018 * screen_spans[str(contrast)]
    for effect, contrast in zip(
        kcnip4_screen["beta_effect"],
        kcnip4_screen["contrast"],
    )
]

p_sfig6h = (
    gg.ggplot()
    + gg.geom_blank(
        data=screen_x_bounds,
        mapping=gg.aes(x="x_bound", y="y_bound"),
        inherit_aes=False,
    )
    + gg.geom_hex(
        data=screen_other,
        mapping=gg.aes(
            x="alpha_effect",
            y="beta_effect",
        ),
        inherit_aes=False,
        bins=30,
        colour="white",
        linewidth=0.10,
    )
    + gg.geom_abline(
        slope=1,
        intercept=0,
        linetype="dotted",
        colour="#9A9A9A",
        linewidth=0.35,
    )
    + gg.geom_hline(
        yintercept=0,
        linetype="dashed",
        colour="#777777",
        linewidth=0.35,
    )
    + gg.geom_point(
        data=screen_highlight,
        mapping=gg.aes(
            x="alpha_effect",
            y="beta_effect",
            colour="display_category",
        ),
        inherit_aes=False,
        size=1.65,
        alpha=0.90,
    )
    + gg.geom_vline(
        xintercept=0,
        linetype="dashed",
        colour="#777777",
        linewidth=0.35,
    )
    + gg.geom_point(
        data=kcnip4_screen,
        mapping=gg.aes(x="alpha_effect", y="beta_effect"),
        inherit_aes=False,
        colour="#D73027",
        size=3.1,
    )
    + gg.geom_text(
        data=kcnip4_screen,
        mapping=gg.aes(x="label_x", y="label_y", label="gene"),
        inherit_aes=False,
        colour="#B2182B",
        size=2.9,
        hjust=0,
    )
    + gg.facet_wrap("contrast", ncol=1, scales="free")
    + gg.scale_colour_manual(
        values=category_colours,
        breaks=category_order,
    )
    + gg.scale_fill_gradient(
        low="#F3F5F6",
        high="#536675",
        guide="none",
    )
    + gg.coord_fixed(ratio=1)
    + gg.labs(
        x="α trajectory mean fitted log2FC",
        y="β trajectory mean fitted log2FC",
        colour=None,
    )
    + gg.theme_minimal(base_size=9)
    + gg.theme(
        legend_position="bottom",
        legend_direction="horizontal",
        legend_title=gg.element_blank(),
        axis_title=gg.element_text(size=8),
        axis_text=gg.element_text(size=7, colour="#555555"),
        panel_grid=gg.element_blank(),
        strip_background=gg.element_blank(),
        strip_text=gg.element_text(size=8, face="bold"),
        panel_spacing=gg.unit(0.22, "in"),
    )
)

save_final_plot(
    p_sfig6h,
    SFIG6_DIR,
    "SFig6h_Kcnip4_cross_lineage_screen",
    4.2,
    7.4,
)
show(str(SFIG6_DIR / "SFig6h_Kcnip4_cross_lineage_screen.png"))


In [ ]:
# Record the frozen inputs and regenerated publication outputs.
publication_inputs = [
    TRAJECTORY_INPUT,
    TABLES / "beam.csv.gz",
    TABLES / "fig2g.tsv",
    TABLES / "screen_control.tsv.gz",
    TABLES / "sfig6g.tsv",
    TABLES / "sfig6g_tests.tsv",
    TABLES / "sfig6h.tsv",
]
publication_outputs = sorted(
    list(FIG2_DIR.glob("*.png"))
    + list(FIG2_DIR.glob("*.pdf"))
    + list(SFIG6_DIR.glob("*.png"))
    + list(SFIG6_DIR.glob("*.pdf"))
)
manifest = {
    "purpose": "Human-reviewed follow-up of the agent analysis",
    "source_notebook": "agent.ipynb",
    "trajectory_refit": False,
    "beam_refit": False,
    "tradeseq_refit": False,
    "seeds": {"analysis": SEED, "DDRTree": DDRTREE_RANDOM_STATE},
    "experimental_unit": "embryo for genotype summaries",
    "inputs": {
        path.name: {
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in publication_inputs
    },
    "outputs": {
        str(path.relative_to(OUTPUT_ROOT)): {
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in publication_outputs
    },
}
manifest_path = OUTPUT_ROOT / "analysis_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Manifest:", manifest_path)


## Interpretation

The supervised DDRTree supplies a common alpha/beta developmental coordinate.
BEAM identifies branch-associated expression programs, while the separate
condition-aware screen ranks genotype-dependent temporal perturbation within
that coordinate. Genotype effects are strongest in terminal hormone-output
genes; canonical lineage-TF programs are comparatively preserved. Embryos,
not cells, are the experimental units for genotype summaries.
